# WriteWise — Matched Phase B Translation Comparison

This fifth experiment tests whether translation clips useful strokes in tightly cropped
characters. A stress probe on 1,000 training images found that 21.6% lost more than
5% of their ink under a maximum allowed shift toward the bottom-right. This is not
the average random training transformation and does not establish an accuracy benefit.
Keep the adaptive-BN checkpoint with 79.14% test accuracy as the current saved best.

**Use a fresh Colab T4 runtime. Run Sections 1–7, then stop.**
Reuse run three's `stage1_phase_a_best.keras`; do not repeat Phase A. Train two branches:
- `translation_5pct`: the current ±5% translation in both directions.
- `no_translation`: a zero translation factor; rotation and zoom remain active.

Both branches use adaptive BN in the upper ~30% of the backbone. Lower layers stay
frozen. Starting weights, fresh Adam optimizers, full shuffling, augmentation seeds,
rotation, zoom, white borders, 52 classes, batch size, split and learning rate match.
Keeping the zero-factor translation layer preserves the other layers' random sequences.
The control is a fresh restart and may not exactly reproduce run four on different hardware.

Outputs go to a new comparison directory. Select using validation loss, then macro F1
and accuracy; exact ties keep the current ±5% translation. Preserve Phase A if a branch
does not improve it. Earlier test results are historical, not selection inputs here.
The 90% target must still be measured; this experiment does not guarantee it.


## 1. Setup & Source Run

Connect Drive and locate run three's Phase A checkpoint. Change `SOURCE_RUN_DIR` only if needed.


In [ ]:
# Mount Google Drive for data access and persistent checkpoint saving
from google.colab import drive

drive.mount("/content/drive")

import glob
import hashlib
import json
import os
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# Configuration — set checkpoint directory in Drive
DRIVE_DIR = "/content/drive/MyDrive/writewise/training"
RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
)
CHECKPOINT_DIR = f"{DRIVE_DIR}/checkpoints/stage1_translation_comparison/{RUN_ID}"
RUN_FINAL_TEST = (
    False  # Enable only after choosing the configuration on validation data.
)
# This is run three's frozen-backbone Phase A checkpoint, not stage1_best.keras.
SOURCE_RUN_DIR = f"{DRIVE_DIR}/checkpoints/stage1_v2/20261018T074232Z-e42906"
SOURCE_PHASE_A = f"{SOURCE_RUN_DIR}/stage1_phase_a_best.keras"
FINAL_VARIANT = None  # Choose after comparing validation reports; leave test disabled.
if not Path(SOURCE_PHASE_A).is_file():
    raise FileNotFoundError(
        f"Missing shared Phase A checkpoint: {SOURCE_PHASE_A}. "
        "Set SOURCE_RUN_DIR to run three's folder in Drive."
    )
SOURCE_PHASE_A_SHA256 = hashlib.sha256(Path(SOURCE_PHASE_A).read_bytes()).hexdigest()
os.makedirs(CHECKPOINT_DIR, exist_ok=False)

# High-performance local caching on Colab local SSD (/content/data/processed):
LOCAL_DATA_DIR = "/content/data/processed"

# Look for processed.zip in common locations (Drive or uploaded directly to Colab)
candidate_zip_paths = [
    "/content/processed.zip",
    f"{DRIVE_DIR}/data/processed.zip",
    f"{DRIVE_DIR}/processed.zip",
    "/content/drive/MyDrive/writewise/data/processed.zip",
    "/content/drive/MyDrive/writewise/processed.zip",
    "/content/drive/MyDrive/processed.zip",
]

zip_found = None
for p in candidate_zip_paths:
    if os.path.exists(p):
        zip_found = p
        break


def resolve_processed_dir(candidates):
    """Select a data root by its split folders, not by substrings in its path."""
    for candidate in candidates:
        root = Path(candidate)
        if root.name == "train":
            root = root.parent
        for option in (root, root / "processed"):
            if all((option / split).is_dir() for split in ("train", "val", "test")):
                return str(option)
    return None


DATA_DIR = resolve_processed_dir([LOCAL_DATA_DIR])
if DATA_DIR is None and zip_found:
    print(f"Extracting {zip_found} to {LOCAL_DATA_DIR}...")
    with zipfile.ZipFile(zip_found, "r") as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    DATA_DIR = resolve_processed_dir([LOCAL_DATA_DIR])
if DATA_DIR is None:
    DATA_DIR = resolve_processed_dir([f"{DRIVE_DIR}/data/processed"])
if DATA_DIR is None:
    found_dirs = sorted(
        glob.glob("/content/drive/MyDrive/**/processed", recursive=True)
    )
    found_dirs += sorted(glob.glob("/content/drive/MyDrive/**/train", recursive=True))
    DATA_DIR = resolve_processed_dir(found_dirs)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find processed CCC train/val/test folders. Upload processed.zip or set DRIVE_DIR."
    )

print(f"Active DATA_DIR:       {DATA_DIR}")
print(f"Active CHECKPOINT_DIR: {CHECKPOINT_DIR}")

In [ ]:
import numpy as np
import tensorflow as tf

SEED = 42
INPUT_SIZE = 96
BATCH_SIZE = 32
PHASE_B_EPOCHS = 50
PHASE_B_LEARNING_RATE = 1e-5
EARLY_STOPPING_PATIENCE = 5
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"GPU available: {len(gpus) > 0}")
if gpus:
    print(f"Using GPU device: {gpus[0].name}")
else:
    print("WARNING: Select a T4 GPU in Runtime -> Change runtime type before training.")

## 2. Load the Same Training & Validation Data

Check class mappings and counts against the source run; record the comparison settings.


In [ ]:
# CCC class ordering is shared by all splits and matches classes.json.
# Never independently enumerate the classes present in validation or test.
CLASS_NAMES = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}
metadata_path = Path(DATA_DIR) / "classes.json"
if metadata_path.exists():
    class_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if (
        class_metadata["classes"] != CLASS_NAMES
        or class_metadata["class_to_idx"] != CLASS_TO_IDX
    ):
        raise ValueError(
            "classes.json does not match the canonical 52-class CCC mapping"
        )


def load_split(split_dir: str) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """Load compact images and encode labels using the same mapping in every split."""
    split_path = Path(split_dir)
    images_file = split_path / "images.npy"
    labels_file = split_path / "labels.npy"
    if images_file.exists() and labels_file.exists():
        images = np.load(images_file, allow_pickle=False)
        raw_labels = np.load(labels_file, allow_pickle=False)
        if raw_labels.ndim != 1:
            raise ValueError("Labels must be a one-dimensional array")
        labels = [str(label) for label in raw_labels]
    else:
        if images_file.exists() or labels_file.exists():
            raise ValueError("Combined split requires both images.npy and labels.npy")
        samples = sorted(split_path.glob("*.npy"))
        labels = [sample.stem.split("_")[0] for sample in samples]
        images = np.array([np.load(sample, allow_pickle=False) for sample in samples])

    if len(images) != len(labels):
        raise ValueError(f"Image/label count mismatch in {split_path}")
    if not labels:
        raise ValueError(f"Empty split: {split_path}")
    if (
        images.dtype != np.uint8
        or images.ndim != 3
        or images.shape[1:] != (INPUT_SIZE, INPUT_SIZE)
    ):
        raise ValueError(
            f"Expected uint8 images with shape (N, 96, 96), got {images.dtype} {images.shape}"
        )
    unknown = sorted(set(labels) - set(CLASS_TO_IDX))
    if unknown:
        raise ValueError(f"Unknown labels in {split_path}: {unknown}")
    encoded = np.array([CLASS_TO_IDX[label] for label in labels], dtype=np.int32)
    return images, encoded, CLASS_NAMES.copy()


print("Loading training data...")
X_train, y_train, class_names = load_split(f"{DATA_DIR}/train")
print("Loading validation data...")
X_val, y_val, val_classes = load_split(f"{DATA_DIR}/val")
NUM_CLASSES = len(class_names)
train_counts = np.bincount(y_train, minlength=NUM_CLASSES)
val_counts = np.bincount(y_val, minlength=NUM_CLASSES)
if np.any(train_counts == 0):
    raise ValueError(
        f"Training has no samples for: {[class_names[i] for i in np.flatnonzero(train_counts == 0)]}"
    )

print(f"Train: {X_train.shape}, {X_train.nbytes / 1e6:.1f} MB; Val: {X_val.shape}")
print("Class | train | val")
for name, train_count, val_count in zip(class_names, train_counts, val_counts):
    print(f"{name:>5} | {train_count:>5} | {val_count:>3}")
missing_val = [class_names[i] for i in np.flatnonzero(val_counts == 0)]
rare_train = [class_names[i] for i in np.flatnonzero(train_counts < 30)]
if missing_val:
    print(f"WARNING: validation cannot measure these classes: {missing_val}")
if rare_train:
    print(f"WARNING: fewer than 30 training samples for: {rare_train}")

source_config = json.loads(
    (Path(SOURCE_RUN_DIR) / "run_config.json").read_text(encoding="utf-8")
)
expected_source = {
    "experiment": "run3_full_shuffle",
    "seed": SEED,
    "input_size": INPUT_SIZE,
    "batch_size": BATCH_SIZE,
    "augmentation_fill": 1.0,
    "class_weighting": False,
    "class_names": class_names,
    "train_counts": train_counts.tolist(),
    "val_counts": val_counts.tolist(),
}
for key, expected in expected_source.items():
    if source_config.get(key) != expected:
        raise ValueError(
            f"Source run/data mismatch for {key}; use run three's data and metadata"
        )

run_config = {
    "version": 1,
    "experiment": "run5_translation_comparison",
    "run_id": RUN_ID,
    "source_run_id": source_config["run_id"],
    "source_phase_a_checkpoint": SOURCE_PHASE_A,
    "source_phase_a_sha256": SOURCE_PHASE_A_SHA256,
    "seed": SEED,
    "tensorflow_version": tf.__version__,
    "source_tensorflow_version": source_config["tensorflow_version"],
    "class_names": class_names,
    "class_to_idx": CLASS_TO_IDX,
    "train_counts": train_counts.tolist(),
    "val_counts": val_counts.tolist(),
    "input_size": INPUT_SIZE,
    "batch_size": BATCH_SIZE,
    "shuffle_buffer_size": len(y_train),
    "reshuffle_each_iteration": True,
    "shuffle_before_normalization": True,
    "augmentation_fill": 1.0,
    "augmentation_seeds": [SEED + 1, SEED + 2, SEED + 3],
    "augmentation_map_parallel_calls": 1,
    "class_weighting": False,
    "bn_frozen": False,
    "translation_factors": {"translation_5pct": 0.05, "no_translation": 0.0},
    "phase_b_learning_rate": PHASE_B_LEARNING_RATE,
    "phase_b_max_epochs": PHASE_B_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_monitor": "val_loss",
    "comparison_order": ["translation_5pct", "no_translation"],
    "selection_order": [
        "val_loss",
        "macro_f1",
        "accuracy",
        "prefer_current_translation_on_tie",
    ],
    "matched_phase_b_restart": True,
}
(Path(CHECKPOINT_DIR) / "run_config.json").write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)
print(f"Shared Phase A: {SOURCE_PHASE_A}")
print(f"Comparison output: {CHECKPOINT_DIR}")

## 3. Matched Data Preparation

Use full shuffling and matching rotation/zoom seeds. Vary only the translation factor:
0.05 for the control and 0.0 for the candidate. Validation preparation is unchanged.


In [ ]:
def make_augmentation(translation_factor):
    """Create fresh seeded generators for each matched Phase B restart."""
    return tf.keras.Sequential(
        [
            tf.keras.layers.RandomRotation(
                seed=SEED + 1,
                factor=15 / 360,  # ±15 degrees ceiling
                fill_mode="constant",
                fill_value=1.0,  # white background in [-1, 1] range
            ),
            tf.keras.layers.RandomZoom(
                seed=SEED + 2,
                height_factor=(-0.1, 0.1),
                width_factor=(-0.1, 0.1),
                fill_mode="constant",
                fill_value=1.0,
            ),
            tf.keras.layers.RandomTranslation(
                seed=SEED + 3,
                height_factor=translation_factor,
                width_factor=translation_factor,
                fill_mode="constant",
                fill_value=1.0,
            ),
        ]
    )


@tf.function
def transform_sample(image, label):
    """Streamed on-the-fly conversion from (96, 96) uint8 to (96, 96, 3) float32 in [-1, 1]."""
    # Grayscale -> 3-channel (MobileNetV2 RGB requirement)
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    # Normalize to [-1, 1]
    image = (tf.cast(image, tf.float32) / 127.5) - 1.0
    # One-hot encode label
    label_onehot = tf.one_hot(label, NUM_CLASSES)
    return image, label_onehot


def build_dataset(
    images: np.ndarray, labels: np.ndarray, augment=False, augmentation=None
):
    """Build a memory-efficient tf.data.Dataset using on-the-fly batch transformation."""
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if augment:
        # Full shuffle while images are still compact uint8 arrays.
        ds = ds.shuffle(
            buffer_size=len(labels), seed=SEED, reshuffle_each_iteration=True
        )

    # Streaming transform (only processes 32 samples at a time in RAM)
    ds = ds.map(transform_sample, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    if augment:
        if augmentation is None:
            raise ValueError("Pass a fresh augmentation layer for each training branch")
        ds = ds.map(
            lambda x, y: (augmentation(x, training=True), y),
            num_parallel_calls=1,  # Keep stateful augmentation draws ordered across branches.
        )

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

## 4. Shared Phase A & Adaptive BN

Validate the shared checkpoint, unfreeze the top ~30% with adaptive BN in both branches,
and preserve the best validation-loss checkpoint. The source checkpoint is read-only.


In [ ]:
def validate_phase_a_model(model):
    """Reject a fine-tuned checkpoint or a different architecture before training."""
    if (
        model.input_shape[1:] != (INPUT_SIZE, INPUT_SIZE, 3)
        or model.output_shape[-1] != NUM_CLASSES
    ):
        raise ValueError("Phase A checkpoint must have 96x96 RGB input and 52 outputs")
    if not isinstance(
        model.layers[-2], tf.keras.layers.GlobalAveragePooling2D
    ) or not isinstance(model.layers[-1], tf.keras.layers.Dense):
        raise TypeError(
            "Phase A checkpoint must preserve the flat backbone -> GAP -> Dense layout"
        )
    if any(layer.trainable and layer.weights for layer in model.layers[:-2]):
        raise ValueError(
            "Use the frozen-backbone Phase A checkpoint, not a Phase B checkpoint"
        )


def configure_bn_policy(model, freeze_bn):
    """Unfreeze the same top 30% of backbone layers; vary only upper-layer BN."""
    model.trainable = True
    backbone_layers = model.layers[:-2]
    freeze_until = int(len(backbone_layers) * 0.7)
    for index, layer in enumerate(backbone_layers):
        layer.trainable = index >= freeze_until and not (
            freeze_bn and isinstance(layer, tf.keras.layers.BatchNormalization)
        )


def phase_b_callbacks(output_dir, phase_a_loss):
    output_dir = Path(output_dir)
    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(output_dir / "stage1_best.keras"),
            monitor="val_loss",
            save_best_only=True,
            initial_value_threshold=phase_a_loss,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(str(output_dir / "phase_b_history.csv")),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def run_translation_variant(name, translation_factor):
    """Load shared Phase A weights and restart both data RNGs and Adam for each branch."""
    if (
        hashlib.sha256(Path(SOURCE_PHASE_A).read_bytes()).hexdigest()
        != SOURCE_PHASE_A_SHA256
    ):
        raise RuntimeError("The source Phase A checkpoint changed; stop the comparison")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    branch_model = tf.keras.models.load_model(SOURCE_PHASE_A, compile=False)
    validate_phase_a_model(branch_model)
    digest = hashlib.sha256()
    for weight in branch_model.get_weights():
        digest.update(np.ascontiguousarray(weight).tobytes())
    initial_weight_sha256 = digest.hexdigest()
    configure_bn_policy(branch_model, False)
    branch_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=PHASE_B_LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    augmentation = make_augmentation(translation_factor)
    train_ds = build_dataset(X_train, y_train, augment=True, augmentation=augmentation)
    val_ds = build_dataset(X_val, y_val, augment=False)
    phase_a_loss = float(
        branch_model.evaluate(val_ds, verbose=0, return_dict=True)["loss"]
    )
    if not np.isfinite(phase_a_loss):
        raise RuntimeError("Shared Phase A validation loss is not finite")
    output_dir = Path(CHECKPOINT_DIR) / name
    output_dir.mkdir(parents=True, exist_ok=False)
    checkpoint = str(output_dir / "stage1_best.keras")
    # Preserve Phase A if this branch never improves its validation loss.
    branch_model.save(checkpoint)
    branch_config = {
        **run_config,
        "variant": name,
        "bn_frozen": False,
        "translation_factor": translation_factor,
        "source_phase_a_sha256": SOURCE_PHASE_A_SHA256,
        "initial_weight_sha256": initial_weight_sha256,
        "phase_a_val_loss": phase_a_loss,
    }
    (output_dir / "run_config.json").write_text(
        json.dumps(branch_config, indent=2), encoding="utf-8"
    )
    print(f"Training {name}; translation ±{translation_factor:.0%}; upper BN adaptive")
    history = branch_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE_B_EPOCHS,
        callbacks=phase_b_callbacks(output_dir, phase_a_loss),
    ).history
    best_b_loss = float(min(history["val_loss"]))
    selected_phase = (
        "B" if np.isfinite(best_b_loss) and best_b_loss < phase_a_loss else "A"
    )
    result = {
        "variant": name,
        "bn_frozen": False,
        "translation_factor": translation_factor,
        "checkpoint": checkpoint,
        "phase_a_val_loss": phase_a_loss,
        "initial_weight_sha256": initial_weight_sha256,
        "selected_phase": selected_phase,
        "history": history,
    }
    (output_dir / "training_history.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    del branch_model, train_ds, val_ds, augmentation
    return result


def select_validation_winner(summaries):
    """Use validation loss first, macro F1 then accuracy; prefer current translation on an exact tie."""
    return min(
        summaries,
        key=lambda name: (
            summaries[name]["val"]["loss"],
            -summaries[name]["val"]["macro_f1"],
            -summaries[name]["val"]["accuracy"],
            summaries[name]["translation_factor"] != 0.05,
        ),
    )

## 5. Train Both Translation Variants

Train `translation_5pct` first, then `no_translation`. Each restarts Phase B with matching
weights and fresh optimizer/random generators, up to 50 epochs with patience 5.
This runs two Phase B trainings and reuses Phase A.


In [ ]:
variant_results = {}
for variant_name, translation_factor in (
    ("translation_5pct", 0.05),
    ("no_translation", 0.0),
):
    variant_results[variant_name] = run_translation_variant(
        variant_name, translation_factor
    )
if len({r["initial_weight_sha256"] for r in variant_results.values()}) != 1:
    raise RuntimeError("Branches did not start from identical weights")
print("Both matched Phase B branches finished. Continue to the validation reports.")

## 6. Training Curves

Compare validation accuracy and loss across the two variants.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, result in variant_results.items():
    history = result["history"]
    epochs = np.arange(1, len(history["val_loss"]) + 1)
    axes[0].plot(epochs, history["val_accuracy"], label=name)
    axes[1].plot(epochs, history["val_loss"], label=name)
axes[0].set(title="Validation accuracy", xlabel="Phase B epoch", ylabel="Accuracy")
axes[1].set(title="Validation loss", xlabel="Phase B epoch", ylabel="Loss")
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(Path(CHECKPOINT_DIR) / "training_curves.png", dpi=150)
plt.show()
plt.close(fig)
(Path(CHECKPOINT_DIR) / "training_history.json").write_text(
    json.dumps(variant_results, indent=2), encoding="utf-8"
)

## 7. Clean Diagnostics & Validation Comparison

Reload both best checkpoints, save clean training/validation reports, and recommend by validation loss. No test images are loaded.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


def evaluate_clean_split(eval_model, images, labels, split_name, output_dir):
    """Evaluate fixed weights with no augmentation or shuffling; keep all 52 labels."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    clean_ds = build_dataset(images, labels, augment=False)
    # Use the training loss path to retain stable cross-entropy from logits.
    evaluation = eval_model.evaluate(clean_ds, verbose=1, return_dict=True)
    predictions = eval_model.predict(clean_ds, verbose=1)
    predicted_labels = np.argmax(predictions, axis=1)
    label_indices = list(range(NUM_CLASSES))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predicted_labels,
        labels=label_indices,
        average="macro",
        zero_division=0,
    )
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    metrics = {
        "samples": len(labels),
        "loss": float(evaluation["loss"]),
        "accuracy": float(accuracy_score(labels, predicted_labels)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1),
        "missing_classes": [class_names[i] for i in np.flatnonzero(counts == 0)],
    }
    report = classification_report(
        labels,
        predicted_labels,
        labels=label_indices,
        target_names=class_names,
        zero_division=0,
    )
    (output_dir / f"{split_name}_classification_report.txt").write_text(
        f"Stage 1 — {split_name} (saved checkpoint, no augmentation)\n"
        f"Accuracy: {metrics['accuracy']:.4f} | Macro F1: {f1:.4f}\n\n{report}",
        encoding="utf-8",
    )
    matrix = confusion_matrix(labels, predicted_labels, labels=label_indices)
    np.savetxt(
        output_dir / f"{split_name}_confusion_matrix.csv",
        matrix,
        delimiter=",",
        fmt="%d",
    )
    fig, ax = plt.subplots(figsize=(16, 14))
    heatmap = ax.imshow(matrix, interpolation="nearest", cmap="Blues")
    fig.colorbar(heatmap, ax=ax)
    ax.set(
        title=f"{split_name}: saved checkpoint accuracy {metrics['accuracy']:.2%}",
        xlabel="Predicted label",
        ylabel="True label",
        xticks=label_indices,
        yticks=label_indices,
    )
    ax.set_xticklabels(class_names, rotation=90, fontsize=6)
    ax.set_yticklabels(class_names, fontsize=6)
    fig.tight_layout()
    fig.savefig(output_dir / f"{split_name}_confusion_matrix.png", dpi=150)
    plt.close(fig)
    print(
        f"{split_name}: loss={metrics['loss']:.4f}, "
        f"accuracy={metrics['accuracy']:.2%}, macro F1={f1:.4f}"
    )
    if metrics["missing_classes"]:
        print(
            f"No examples for {metrics['missing_classes']}; macro metrics include "
            "all classes with zero for unsupported classes."
        )
    return metrics


comparison_summaries = {}
for name, result in variant_results.items():
    diagnostic_dir = Path(CHECKPOINT_DIR) / name / "diagnostics"
    diagnostic_model = tf.keras.models.load_model(result["checkpoint"], compile=False)
    diagnostic_model.compile(loss="categorical_crossentropy", metrics=["accuracy"])
    train_metrics = evaluate_clean_split(
        diagnostic_model, X_train, y_train, "train_clean", diagnostic_dir
    )
    val_metrics = evaluate_clean_split(
        diagnostic_model, X_val, y_val, "val", diagnostic_dir
    )
    summary = {
        "experiment": "run5_translation_comparison",
        "run_id": RUN_ID,
        "variant": name,
        "bn_frozen": result["bn_frozen"],
        "translation_factor": result["translation_factor"],
        "checkpoint": result["checkpoint"],
        "selected_phase": result["selected_phase"],
        "source_phase_a_sha256": SOURCE_PHASE_A_SHA256,
        "initial_weight_sha256": result["initial_weight_sha256"],
        "macro_average": "all_classes_including_unsupported",
        "class_names": class_names,
        "train_clean": train_metrics,
        "val": val_metrics,
        "generalization_gap_percentage_points": 100
        * (train_metrics["accuracy"] - val_metrics["accuracy"]),
    }
    (diagnostic_dir / "validation_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    comparison_summaries[name] = summary
    del diagnostic_model

recommended_variant = select_validation_winner(comparison_summaries)
comparison_summary = {
    "experiment": "run5_translation_comparison",
    "run_id": RUN_ID,
    "source_run_id": run_config["source_run_id"],
    "selection_order": run_config["selection_order"],
    "recommended_variant": recommended_variant,
    "variants": comparison_summaries,
}
(Path(CHECKPOINT_DIR) / "comparison_summary.json").write_text(
    json.dumps(comparison_summary, indent=2), encoding="utf-8"
)
print("\nVariant              | Val loss | Accuracy | Macro F1")
for name, summary in comparison_summaries.items():
    val = summary["val"]
    print(
        f"{name:20} | {val['loss']:.4f}   | {val['accuracy']:.2%}   | {val['macro_f1']:.4f}"
    )
print(f"Validation recommendation: {recommended_variant}")
print(
    f"Share {Path(CHECKPOINT_DIR) / 'comparison_summary.json'} before enabling final testing."
)

## 8. Optional Final Test — After Reviewing the Comparison

Share `comparison_summary.json` first. Compare validation results with the current best
adaptive-BN run before selecting a new candidate. After selecting on validation, add a new cell:
```python
FINAL_VARIANT = "no_translation"  # or "translation_5pct", based on validation
RUN_FINAL_TEST = True
```
Run that cell and this evaluation cell once. Do not rerun setup or training.
Do not evaluate both candidates on the test set to select the policy. Retain all earlier results.


In [ ]:
if not RUN_FINAL_TEST:
    raise RuntimeError(
        "Final test is disabled. Select the configuration on validation first, then set RUN_FINAL_TEST = True."
    )

if FINAL_VARIANT not in ("translation_5pct", "no_translation"):
    raise RuntimeError("Choose FINAL_VARIANT after comparing validation results")
selected_result = variant_results[FINAL_VARIANT]
checkpoint_path = selected_result["checkpoint"]
selected_phase = selected_result["selected_phase"]
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)

eval_dir = f"{CHECKPOINT_DIR}/{FINAL_VARIANT}/evaluation"
os.makedirs(eval_dir, exist_ok=True)

print("Loading best model checkpoint for evaluation...")
eval_model = tf.keras.models.load_model(checkpoint_path)

print("Loading the official CCC test split...")
X_test, y_test, test_classes = load_split(f"{DATA_DIR}/test")
print(f"  Test samples: {len(X_test)}")
print(f"  Classes:      {len(test_classes)}")
if len(X_test) != 19133:
    raise ValueError(
        f"Expected the official 19,133-character CCC test set, got {len(X_test)}"
    )


# RAM-safe streamed test dataset (no massive float32 arrays in RAM!)
@tf.function
def test_preprocess(image):
    image = tf.expand_dims(image, -1)
    image = tf.repeat(image, 3, axis=-1)
    return (tf.cast(image, tf.float32) / 127.5) - 1.0


test_ds = tf.data.Dataset.from_tensor_slices(X_test)
test_ds = (
    test_ds.map(test_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

print("Running test set inference...")
test_predictions = eval_model.predict(test_ds, verbose=1)
y_pred = np.argmax(test_predictions, axis=1)

# Metrics
acc = float(accuracy_score(y_test, y_pred))
prec = float(precision_score(y_test, y_pred, average="macro", zero_division=0))
rec = float(recall_score(y_test, y_pred, average="macro", zero_division=0))
f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0))

passed = acc >= 0.90

print()
print("=" * 65)
print("  STAGE 1 FINAL EVALUATION METRICS (CCC TEST SET - 19,133 SAMPLES)")
print("=" * 65)
print(f"  Overall Accuracy:  {acc:.4f} ({acc * 100:.2f}%)")
print(f"  Macro Precision:   {prec:.4f}")
print(f"  Macro Recall:      {rec:.4f}")
print(f"  Macro F1-Score:    {f1:.4f}")
print("  PRD §11 Target:    >= 90.0%")
print(
    f"  Status:            {'✅ PASS (Target Achieved!)' if passed else '❌ BELOW TARGET'}"
)
print("=" * 65)

# Save Classification Report
report = classification_report(
    y_test,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=test_classes,
    zero_division=0,
)
with open(f"{eval_dir}/classification_report.txt", "w", encoding="utf-8") as f:
    f.write("Stage 1 Evaluation — Classification Report\n")
    f.write(f"Accuracy: {acc:.4f} | F1: {f1:.4f}\n\n")
    f.write(report)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=list(range(NUM_CLASSES)))
np.savetxt(f"{eval_dir}/confusion_matrix.csv", cm, delimiter=",", fmt="%d")

plt.figure(figsize=(16, 14))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title(f"Stage 1 Confusion Matrix (Test Accuracy: {acc * 100:.1f}%)", fontsize=14)
plt.colorbar()
ticks = list(range(len(test_classes)))
plt.xticks(ticks, test_classes, rotation=90, fontsize=6)
plt.yticks(ticks, test_classes, fontsize=6)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.tight_layout()
plt.savefig(f"{eval_dir}/confusion_matrix.png", dpi=150)
plt.show()

# Save summary JSON
summary = {
    "experiment": "run5_translation_comparison",
    "variant": FINAL_VARIANT,
    "bn_frozen": False,
    "translation_factor": selected_result["translation_factor"],
    "run_id": RUN_ID,
    "selected_phase": selected_phase,
    "class_names": class_names,
    "checkpoint": checkpoint_path,
    "test_samples": len(X_test),
    "accuracy": round(acc, 6),
    "macro_precision": round(prec, 6),
    "macro_recall": round(rec, 6),
    "macro_f1": round(f1, 6),
    "passes_prd_target": passed,
}
with open(f"{eval_dir}/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nAll artifacts saved to Drive at: {eval_dir}")